# Data Cleaning and Preprocessing

## Objective

In this notebook, we will prepare the datasets for analysis by:

- Loading all datasets
- Converting data types
- Handling missing values
- Checking duplicate records
- Validating primary keys
- Validating foreign keys
- Preparing clean datasets for feature engineering

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",100)

In [2]:
customers = pd.read_csv("../Data/olist_customers_dataset.csv")
orders = pd.read_csv("../Data/olist_orders_dataset.csv")
order_items = pd.read_csv("../Data/olist_order_items_dataset.csv")
payments = pd.read_csv("../Data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/olist_order_reviews_dataset.csv")
products = pd.read_csv("../Data/olist_products_dataset.csv")
sellers = pd.read_csv("../Data/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../Data/olist_geolocation_dataset.csv")
category_translation = pd.read_csv("../Data/product_category_name_translation.csv")

# Convert Date Columns

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

In [4]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

# Check Missing Values

In [5]:
for name, df in {
    "Orders": orders,
    "Products": products,
    "Reviews": reviews
}.items():
    
    print("="*60)
    print(name)
    print("="*60)
    
    missing = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Percentage": round((df.isnull().sum()/len(df))*100,2)
    })

    display(missing[missing["Missing Values"]>0])

Orders


,Missing Values,Percentage
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


Products


,Missing Values,Percentage
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


Reviews


,Missing Values,Percentage
review_comment_title,87656,88.34
review_comment_message,58247,58.70


# Check Duplicate Records

In [6]:
for name, df in {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation
}.items():
    
    print(f"{name:<20}{df.duplicated().sum()}")

Customers           0
Orders              0
Order Items         0
Payments            0
Reviews             0
Products            0
Sellers             0
Geolocation         261831


# Remove Duplicate Records

In [7]:
geolocation = geolocation.drop_duplicates()

In [8]:
geolocation.duplicated().sum()

np.int64(0)

In [9]:
geolocation.shape

(738332, 5)

# Primary Key Validation

In relational databases, every table should have a unique identifier (Primary Key).

This section validates whether the primary keys contain duplicate values.

In [10]:
primary_keys = {
    "Customers": "customer_id",
    "Orders": "order_id",
    "Products": "product_id",
    "Sellers": "seller_id",
    "Category Translation": "product_category_name"
}

for table, key in primary_keys.items():
    
    df = {
        "Customers": customers,
        "Orders": orders,
        "Products": products,
        "Sellers": sellers,
        "Category Translation": category_translation
    }[table]
    
    print(f"{table:<25} Duplicate Primary Keys : {df[key].duplicated().sum()}")

Customers                 Duplicate Primary Keys : 0
Orders                    Duplicate Primary Keys : 0
Products                  Duplicate Primary Keys : 0
Sellers                   Duplicate Primary Keys : 0
Category Translation      Duplicate Primary Keys : 0


# Foreign Key Validation

In [11]:
print("Customer IDs missing from Customers Table :",
      (~orders["customer_id"].isin(customers["customer_id"])).sum())

print("Product IDs missing from Products Table :",
      (~order_items["product_id"].isin(products["product_id"])).sum())

print("Seller IDs missing from Sellers Table :",
      (~order_items["seller_id"].isin(sellers["seller_id"])).sum())

print("Order IDs missing from Orders Table :",
      (~order_items["order_id"].isin(orders["order_id"])).sum())

Customer IDs missing from Customers Table : 0
Product IDs missing from Products Table : 0
Seller IDs missing from Sellers Table : 0
Order IDs missing from Orders Table : 0


# Order Status Distribution

In [12]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

# Missing Values Analysis

In [13]:
orders[orders["order_delivered_customer_date"].isnull()][
    ["order_status"]
].value_counts()

order_status
shipped         1107
canceled         619
unavailable      609
invoiced         314
processing       301
delivered          8
created            5
approved           2
Name: count, dtype: int64

# Review Comment Analysis

In [14]:
reviews[["review_comment_title",
         "review_comment_message"]].isnull().sum()

review_comment_title      87656
review_comment_message    58247
dtype: int64

In [15]:
round(reviews.isnull().mean()*100,2)

review_id                   0.00
order_id                    0.00
review_score                0.00
review_comment_title       88.34
review_comment_message     58.70
review_creation_date        0.00
review_answer_timestamp     0.00
dtype: float64

# Product Missing Values

In [16]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [17]:
products[products["product_category_name"].isnull()].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


# Data Cleaning Strategy

Based on our analysis:

- Remove exact duplicate records from the Geolocation dataset.
- Convert date columns to datetime format.
- Keep missing delivery dates because they represent undelivered or cancelled orders.
- Keep missing review comments because customers are not required to write comments.
- Remove products with missing category information since they cannot be categorized during analysis.
- Fill missing product dimensions using the median.

In [18]:
products.shape

(32951, 9)

In [19]:
products = products.dropna(subset=["product_category_name"])

In [20]:
products.shape

(32341, 9)

In [21]:
products.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              1
product_length_cm             1
product_height_cm             1
product_width_cm              1
dtype: int64

In [22]:
columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in columns:
    products[col].fillna(products[col].median(), inplace=True)

In [23]:
products.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

# Cleaning Product Category Names

In [24]:
products["product_category_name"].nunique()

73

In [25]:
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [27]:
products.head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0,musical_instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0,cool_stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56.0,184.0,2.0,900.0,40.0,8.0,40.0,furniture_decor
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,57.0,163.0,1.0,400.0,27.0,13.0,17.0,home_appliances
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,36.0,1156.0,1.0,600.0,17.0,10.0,12.0,toys


In [28]:
products.shape

(32341, 10)

# Final Missing Value Check

In [29]:
for name, df in {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation
}.items():
    
    print("="*60)
    print(name)
    print("="*60)
    
    print(df.isnull().sum().sum())

Customers
0
Orders
4908
Order Items
0
Payments
0
Reviews
145903
Products
13
Sellers
0
Geolocation
0
